# 2.5D DC — SimPEG vanilla (GN + Pardiso) vs **dctorch** (L-BFGS + cuDSS)

Mismo sintético `true_mod.npy` (30×208), misma regularización L2, **β0 heredado del
vanilla**, mismo cooling (/2) y mismo criterio de parada (χ²≤1): la comparación justa
del notebook histórico, ahora sobre el stack nuevo.

**Qué es dctorch** (2026-07-13): un paquete que extiende **SimPEG 0.25 stock** — nada de
site-packages parchado. SimPEG define la física (los mapas de ensamblaje se extraen de su
propia simulación y se validan a 1e-10 contra `getA` en cada construcción) y dirige el
protocolo (`BaseInversion` + directives oficiales); el runtime (forward+gradiente) es torch
puro en GPU vía cuDSS (nvmath). Hechos medidos que este notebook reproduce:
- **paridad del forward** vs stock a ~1e-15 (sección 3);
- **persist L-BFGS** (default): la curvatura sobrevive el cooling de β → ~2× menos closures;
- **F32 cadena completa**: mitad de memoria, ~1.3-2× tiempo, modelo indistinguible.

**Kernel: `simpeg311`** (python 3.11, simpeg 0.25.2, torch 2.11 cu128, nvmath 1.0).

## 0. Chequeo de ipywidgets (correr PRIMERO — habilita el visor de la última sección)

In [ ]:
HAS_WIDGETS = False
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
    print("ipywidgets OK — el visor interactivo funcionará")
except Exception as e:
    print("sin ipywidgets: el visor mostrará solo la última iteración |", e)

## 1. Imports y parámetros

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # torch(libomp) + MKL/Pardiso(libiomp5)
                                                        # en un proceso: sin esto el kernel CRASHEA
import sys, time
sys.path.insert(0, "..")                    # raíz del repo -> paquete dctorch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import torch
from pymatsolver import Pardiso
from simpeg import (maps, data, data_misfit, regularization, optimization,
                    inverse_problem, inversion, directives)
from simpeg.electromagnetics.static import resistivity as dc
from simpeg.electromagnetics.static.utils.static_utils import generate_dcip_sources_line
from discretize import TensorMesh
from dctorch import TorchDC2D, TorchLBFGS

NKY, NOISE, SEED = 12, 0.03, 1
DEPTH = 30.0
VMIN, VMAX = 30.0, 7000.0
assert torch.cuda.is_available(), "este notebook compara contra GPU"
print(torch.cuda.get_device_name(0))

## 2. Modelo verdadero + survey + malla

La MISMA malla y survey para ambos mundos (con simpeg 0.25 stock ya no hace falta la danza
de "meshes frescos" del fork: nadie contamina caches con tensores).

In [ ]:
model_np = np.load("true_mod.npy")                       # (30, 208) ohm-m
NZ, NX = model_np.shape
l_line, sep = 208.0, 8.0
dx, dz = l_line / NX, DEPTH / NZ
xx = np.linspace(0, l_line, int(l_line))
topo_2d = np.unique(np.c_[xx, np.zeros_like(xx)], axis=0)
src_list = generate_dcip_sources_line("dipole-dipole", "volt", "2D",
                                      np.r_[0.0, l_line], topo_2d, 10, sep)
survey = dc.survey.Survey(src_list)
mesh = TensorMesh([[(dx, 12, -1.2), (dx, NX), (dx, 12, 1.2)],
                   [(dz, 8, -1.2), (dz, NZ)]])
mesh.origin = [-mesh.nodes_x[12], -mesh.nodes_y[-1]]
cc = mesh.cell_centers
core_idx = (cc[:, 0] > 0) & (cc[:, 0] < l_line) & (cc[:, 1] < 0) & (cc[:, 1] > -DEPTH)
m_true = np.log(100.0) * np.ones(mesh.nC)
m_true[core_idx] = np.log(model_np.flatten())
nD = survey.nD


def core_img(m):
    return np.asarray(m)[core_idx].reshape(NZ, NX)


from scipy.ndimage import uniform_filter

def ssim(a_log, b_log, win=7):
    A, B = core_img(a_log).astype(float), core_img(b_log).astype(float)
    L = max(A.max(), B.max()) - min(A.min(), B.min())
    if L == 0:
        return 1.0
    w = min(win, A.shape[0], A.shape[1]); w = w - 1 if w % 2 == 0 else w; w = max(w, 3)
    c1, c2 = (0.01 * L) ** 2, (0.03 * L) ** 2
    ma, mb = uniform_filter(A, w), uniform_filter(B, w)
    va = uniform_filter(A * A, w) - ma ** 2
    vb = uniform_filter(B * B, w) - mb ** 2
    vab = uniform_filter(A * B, w) - ma * mb
    s = ((2 * ma * mb + c1) * (2 * vab + c2)) / ((ma**2 + mb**2 + c1) * (va + vb + c2))
    return float(s.mean())


print(f"mesh {mesh.nC:,} celdas | nD {nD} | celda {dx:.1f}x{dz:.1f} m")

### Datos sintéticos (3% de ruido) y arranque compartido

In [ ]:
sim_data = dc.simulation_2d.Simulation2DNodal(mesh, survey=survey,
                                              rhoMap=maps.ExpMap(mesh),
                                              nky=NKY, solver=Pardiso)
t0 = time.time()
dclean = sim_data.dpred(m_true)
t_fwd_stock = time.time() - t0
rng = np.random.default_rng(SEED)
std = NOISE * np.abs(dclean)
dobs = dclean + std * rng.standard_normal(nD)
dat = data.Data(survey, dobs=dobs, standard_deviation=std)
m0 = np.log(100.0) * np.ones(mesh.nC)
print(f"forward verdad (stock+Pardiso, la referencia CPU): {t_fwd_stock:.1f}s | nD {nD}")


class Recorder(directives.InversionDirective):
    """Guarda modelo/phi_d/tiempo por iteracion (alimenta el visor) + beta0."""
    def initialize(self):
        self.t0 = time.time()
        self.models = [np.array(self.invProb.model)]
        self.phi, self.tt = [], []
        self.beta0 = None

    def endIter(self):
        if self.beta0 is None:
            self.beta0 = float(self.invProb.beta)
        self.models.append(np.array(self.invProb.model))
        self.phi.append(float(self.invProb.phi_d))
        self.tt.append(time.time() - self.t0)

## 3. Verificación de equivalencia (el claim del paper)

Antes de comparar velocidad: el motor dctorch debe calcular **la misma física** que el
SimPEG oficial. Además del assert interno (mapas L_ky validados a 1e-10 contra `getA` en
cada construcción), la paridad del dato completo:

In [ ]:
t0 = time.time()
eng64 = TorchDC2D(mesh, survey, nky=NKY, device="cuda")     # F64, se reusa abajo
t_build = time.time() - t0
d_torch = eng64.dpred(torch.tensor(m_true, device="cuda")).cpu().numpy()
rel = np.linalg.norm(d_torch - dclean) / np.linalg.norm(dclean)
print(f"[build motor] {t_build:.1f}s (una vez por malla/survey)")
print(f"[paridad] dpred dctorch vs stock-Pardiso: rel-err {rel:.2e}  -> {'OK' if rel < 1e-10 else 'REVISAR'}")

## 4. Pierna A — vanilla: Gauss-Newton + Pardiso (SimPEG 0.25 stock, CPU)

El estándar de la industria como referencia FUERTE: `storeJ=True` (J explícita, lo mejor
para GN a este tamaño), Pardiso MKL, β0 por power-iteration con semilla fija.

In [ ]:
sim_gn = dc.simulation_2d.Simulation2DNodal(mesh, survey=survey,
                                            rhoMap=maps.ExpMap(mesh),
                                            nky=NKY, solver=Pardiso, storeJ=True)
dmis_gn = data_misfit.L2DataMisfit(data=dat, simulation=sim_gn)
reg_gn = regularization.WeightedLeastSquares(mesh, alpha_s=0.16, alpha_x=1.0,
                                             alpha_y=1.0, reference_model=m0)
opt_gn = optimization.InexactGaussNewton(maxIter=20, maxIterCG=20)
prob_gn = inverse_problem.BaseInvProblem(dmis_gn, reg_gn, opt_gn)
rec_gn = Recorder()
dirs_gn = [directives.BetaEstimate_ByEig(beta0_ratio=100.0, random_seed=SEED),
           directives.BetaSchedule(coolingFactor=2.0, coolingRate=1),
           directives.TargetMisfit(chifact=1.0), rec_gn]
t0 = time.time()
m_gn = inversion.BaseInversion(prob_gn, directiveList=dirs_gn).run(m0)
t_gn = time.time() - t0
chi_gn = prob_gn.phi_d / nD
print(f"\n[GN vanilla] {t_gn:.0f}s | chi2 {chi_gn:.2f} | beta0 {rec_gn.beta0:.3e} | "
      f"iters {len(rec_gn.phi)} | SSIM vs verdad {ssim(m_gn, m_true):.3f}")

## 5. Pierna B — dctorch: TorchLBFGS + cuDSS (GPU), F64 y F32

Mismo protocolo SimPEG (`BaseInvProblem` + directives oficiales), **β0 heredado del
vanilla**, mismo cooling y target. `persist_inner=True` es el default del optimizador
(la curvatura sobrevive el cooling de β — 2× medido). El sim stock aquí es solo el
contenedor de W/dobs del protocolo: nunca se resuelve.

In [ ]:
def run_dctorch(eng, tag):
    dmis_t = data_misfit.L2DataMisfit(data=dat, simulation=sim_data)
    reg_t = regularization.WeightedLeastSquares(mesh, alpha_s=0.16, alpha_x=1.0,
                                                alpha_y=1.0, reference_model=m0)
    opt_t = TorchLBFGS(engine=eng, maxIter=25, inner_maxiter=30,
                       lower=np.log(1.0), upper=np.log(1e5))   # guard numerico:
                       # exp() no explota en pasos grandes del line search interno
    prob_t = inverse_problem.BaseInvProblem(dmis_t, reg_t, opt_t)
    prob_t.beta = rec_gn.beta0                     # beta0 HEREDADO (comparacion justa)
    rec = Recorder()
    dirs_t = [directives.BetaSchedule(coolingFactor=2.0, coolingRate=1),
              directives.TargetMisfit(chifact=1.0), rec]
    t0 = time.time()
    m_rec = inversion.BaseInversion(prob_t, directiveList=dirs_t).run(m0)
    t = time.time() - t0
    chi = prob_t.phi_d / nD
    print(f"\n[{tag}] {t:.0f}s | chi2 {chi:.2f} | iters {len(rec.phi)} | "
          f"SSIM vs verdad {ssim(m_rec, m_true):.3f}")
    return dict(m=m_rec, rec=rec, t=t, chi=chi)


res64 = run_dctorch(eng64, "dctorch F64")
eng32 = TorchDC2D(mesh, survey, nky=NKY, device="cuda", dtype=torch.float32)
res32 = run_dctorch(eng32, "dctorch F32")

## 6. Convergencia: las DOS gráficas honestas

Por **iteración** puede ganar GN (cada paso GN usa el Hessiano aproximado completo);
por **tiempo de pared** gana el que tenga la iteración barata. Las dos, lado a lado
(la lección del notebook histórico: comparar solo una es hacer trampa).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
legs = [("GN + Pardiso (CPU)", rec_gn, "C0"),
        ("dctorch F64 (GPU)", res64["rec"], "C1"),
        ("dctorch F32 (GPU)", res32["rec"], "C2")]
for ax, xmode in zip(axs, ("iter", "wall")):
    for name, rec, c in legs:
        x = np.arange(1, len(rec.phi) + 1) if xmode == "iter" else np.array(rec.tt)
        ax.semilogy(x, np.array(rec.phi) / nD, "o-", color=c, label=name)
    ax.axhline(1.0, color="k", ls="--", lw=0.8, label="target chi2=1")
    ax.set_xlabel("iteracion (outer)" if xmode == "iter" else "tiempo de pared [s]")
    ax.set_ylabel("chi2 = phi_d / nD")
    ax.grid(alpha=0.3)
axs[0].legend()
axs[0].set_title("por iteracion")
axs[1].set_title("por tiempo de pared")
fig.tight_layout()

print(f"{'pierna':22s} {'t [s]':>7} {'chi2':>6} {'iters':>6} {'SSIM':>6}")
for name, res_m, res_t, res_chi, rec in [
        ("GN vanilla (CPU)", m_gn, t_gn, chi_gn, rec_gn),
        ("dctorch F64 (GPU)", res64["m"], res64["t"], res64["chi"], res64["rec"]),
        ("dctorch F32 (GPU)", res32["m"], res32["t"], res32["chi"], res32["rec"])]:
    print(f"{name:22s} {res_t:7.1f} {res_chi:6.2f} {len(rec.phi):6d} "
          f"{ssim(res_m, m_true):6.3f}")

## 7. Secciones: verdad vs recuperados

In [ ]:
panels = [("verdad", m_true), ("GN vanilla", m_gn),
          ("dctorch F64", res64["m"]), ("dctorch F32", res32["m"])]
fig, axs = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
ext = (0, l_line, -DEPTH, 0)
for ax, (name, mm) in zip(axs, panels):
    im = ax.imshow(np.exp(core_img(mm)), extent=ext, origin="lower",
                   aspect="auto", norm=LogNorm(VMIN, VMAX), cmap="turbo")
    tag = "" if name == "verdad" else f"  |  SSIM {ssim(mm, m_true):.3f}"
    ax.set_title(f"{name}{tag}", fontsize=10)
    ax.set_ylabel("z [m]")
    plt.colorbar(im, ax=ax, pad=0.01, label="rho [ohm-m]")
axs[-1].set_xlabel("x [m]")
fig.tight_layout()

## 8. Visor interactivo por iteración

Mueve el slider: modelo del GN (izq) vs dctorch F64 (der) en la iteración k
(la última disponible de cada pierna si k la excede).

In [ ]:
def show_iter(k=0):
    fig, axs = plt.subplots(1, 2, figsize=(13, 2.8))
    ext = (0, l_line, -DEPTH, 0)
    for ax, (name, rec) in zip(axs, [("GN vanilla", rec_gn),
                                     ("dctorch F64", res64["rec"])]):
        j = min(k, len(rec.models) - 1)
        ax.imshow(np.exp(core_img(rec.models[j])), extent=ext, origin="lower",
                  aspect="auto", norm=LogNorm(VMIN, VMAX), cmap="turbo")
        chi_j = rec.phi[j - 1] / nD if j > 0 else float("nan")
        ax.set_title(f"{name} — iter {j} | chi2 {chi_j:.2f}", fontsize=10)
    plt.show()


if HAS_WIDGETS:
    kmax = max(len(rec_gn.models), len(res64["rec"].models)) - 1
    widgets.interact(show_iter,
                     k=widgets.IntSlider(value=0, min=0, max=kmax, step=1,
                                         description="iteracion"))
else:
    show_iter(999)